# Practical 5 — Continuous Bag of Words (CBOW)

Train a small CBOW model to predict a target word from surrounding context words.

## Aim
Implement the Continuous Bag of Words (CBOW) word-embedding approach using a small text corpus.

## Note
This notebook uses a tiny example corpus to make the mechanics easy to inspect. Embeddings learned from this small corpus are illustrative only; meaningful word representations require much larger and more varied text.

In [ ]:
# 1. Imports
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Lambda, Dense
from tensorflow.keras.utils import to_categorical

In [ ]:
# 2. Prepare and tokenize a small corpus
corpus = [
    "deep learning makes computers intelligent",
    "machine learning helps computers learn",
    "deep learning uses neural networks",
    "neural networks learn patterns from data",
    "machine learning uses data",
    "computers process data efficiently"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
sequences = tokenizer.texts_to_sequences(corpus)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

print("Vocabulary size (including reserved index 0):", vocab_size)
print("Word index:", word_index)
print("Tokenized corpus:", sequences)

In [ ]:
# 3. Create CBOW context-target pairs
window_size = 2
contexts = []
targets = []

for sequence in sequences:
    for i in range(window_size, len(sequence) - window_size):
        context = sequence[i-window_size:i] + sequence[i+1:i+window_size+1]
        target = sequence[i]
        contexts.append(context)
        targets.append(target)

contexts = np.asarray(contexts, dtype="int32")
targets = np.asarray(targets, dtype="int32")

if len(contexts) == 0:
    raise ValueError("No CBOW examples generated. Use longer sentences or a smaller window_size.")

target_labels = to_categorical(targets, num_classes=vocab_size)

print("Context matrix shape:", contexts.shape)
print("Target labels shape:", target_labels.shape)
print("First context token IDs:", contexts[0])
print("First target token ID:", targets[0])

In [ ]:
# 4. Build CBOW model
embedding_dim = 10

model = Sequential([
    Input(shape=(window_size * 2,)),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# 5. Train the model
history = model.fit(
    contexts,
    target_labels,
    epochs=150,
    batch_size=8,
    verbose=0
)
print("Training complete.")

In [ ]:
# 6. Predict the target word from one context
sample_context = contexts[:1]
probabilities = model.predict(sample_context, verbose=0)
predicted_index = int(np.argmax(probabilities[0]))

index_word = {index: word for word, index in word_index.items()}
context_words = [index_word.get(int(i), "<unknown>") for i in sample_context[0]]

print("Context words:", context_words)
print("Actual target:", index_word.get(int(targets[0]), "<unknown>"))
print("Predicted target:", index_word.get(predicted_index, "<unknown>"))

In [ ]:
# 7. Inspect the learned embedding for a word
embedding_matrix = model.layers[0].get_weights()[0]
word = "learning"
word_id = word_index[word]
print(f"Embedding for '{word}' (dimension {embedding_dim}):")
print(embedding_matrix[word_id])

# Optional: plot the loss curve
plt.figure(figsize=(6, 4))
plt.plot(history.history["loss"])
plt.title("CBOW Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Categorical cross-entropy")
plt.show()

## Result
Context-target word pairs are generated from a corpus, a CBOW neural network is trained to predict target words from surrounding context, and a learned word embedding is inspected.